# 06 RAG Pipeline

**Final model stack:**

| Step | Model | Cost | Why |
|---|---|---|---|
| Embeddings (index) | `text-embedding-3-large` | ~$0.11 one-time | Best multilingual retrieval |
| Embeddings (query) | `text-embedding-3-large` | ~$0.000013/query | Same space as index |
| Query Rewriting | `gpt-4o-mini` | ~$0.000015/call | Cheap, fast, excellent instruction following |
| Answer Generation | `gpt-4o-mini` | ~$0.001/answer | Native Bulgarian, practical, direct |

**~$0.001 per conversation turn. Essentially free at personal use scale.**

**System prompt philosophy:**
- Direct, practical answers - not lectures
- Kilograms, not pounds
- Apply the advice immediately - no unnecessary citations
- Talk like a knowledgeable coach, not a textbook

**Pipeline:**
```
User question (BG)
        ↓
  Query Rewriting    ← gpt-4o-mini → optimized EN search query
        ↓
  Dual Retrieval     ← text-embedding-3-large + FAISS (local, free at query time)
        ↓
  Answer Generation  ← gpt-4o-mini → Bulgarian answer
```

In [3]:
import json, os
from pathlib import Path
from typing import Optional
import numpy as np
import faiss
import warnings; warnings.filterwarnings('ignore')

from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt
load_dotenv(override=True)

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
VS_DIR = BACKEND_DIR / 'data' / 'vectorstore'

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

EMBED_MODEL = os.getenv('EMBEDDING_MODEL')
EMBED_DIM = 3072
LLM_MODEL = os.getenv('REWRITING_MODEL')
RETRIEVAL_K = 20
FINAL_K = 5
TEMPERATURE = 0.4

print('Setup complete')
print(f'Embedding: {EMBED_MODEL}')
print(f'LLM: {LLM_MODEL}')

Setup complete
Embedding: text-embedding-3-large
LLM: gpt-4o-mini


## 1. Load FAISS Index + Metadata

In [4]:
index = faiss.read_index(str(VS_DIR / 'henselmans_openai.index'))
metadata = json.loads((VS_DIR / 'henselmans_openai_metadata.json').read_text(encoding='utf-8'))
assert index.ntotal == len(metadata)
print(f'FAISS index: {index.ntotal:,} vectors  ({EMBED_DIM}d)')
print(f'Metadata: {len(metadata):,} records')

FAISS index: 2,098 vectors  (3072d)
Metadata: 2,098 records


## 2. Query Rewriting

In [5]:
@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def rewrite_query(question: str, history: list[dict] = []) -> str:
    """Rewrite any-language question to optimized EN search query for the Henselmans KB."""
    history_ctx = ''
    if history:
        for msg in history[-4:]:
            role = 'User' if msg['role'] == 'user' else 'Assistant'
            history_ctx += f'{role}: {msg["content"][:150]}\n'

    prompt = f"""{f'Recent conversation:{chr(10)}{history_ctx}{chr(10)}' if history_ctx else ''}User question: {question}

Rewrite as a SHORT English search query (5-10 words) for a fitness science knowledge base (Henselmans PTC course).
Reply ONLY with the search query, nothing else."""

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=30,
    )
    return response.choices[0].message.content.strip().strip('"')


# Test
q_test = 'Колко протеин трябва да приемам за мускулна маса?'
rewritten = rewrite_query(q_test)
print(f'Original: {q_test}')
print(f'Rewritten: {rewritten}')

Original: Колко протеин трябва да приемам за мускулна маса?
Rewritten: How much protein for muscle mass?


## 3. Dual Retrieval

In [6]:
def embed_query(text: str) -> np.ndarray:
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    emb = np.array(response.data[0].embedding, dtype='float32').reshape(1, -1)
    faiss.normalize_L2(emb)
    return emb

def retrieve_faiss(query: str, k: int = RETRIEVAL_K) -> list[dict]:
    q_emb = embed_query(query)
    scores, ids = index.search(q_emb, k)
    return [
        {**metadata[idx], 'score': float(s)}
        for s, idx in zip(scores[0], ids[0]) if idx != -1
    ]

def dual_retrieve(original: str, rewritten: str, k: int = RETRIEVAL_K) -> list[dict]:
    """Search with both queries, merge, deduplicate, sort by score."""
    seen = {}
    for r in retrieve_faiss(original, k) + retrieve_faiss(rewritten, k):
        cid = r['chunk_id']
        if cid not in seen or r['score'] > seen[cid]['score']:
            seen[cid] = r
    return sorted(seen.values(), key=lambda x: x['score'], reverse=True)

# Test
candidates = dual_retrieve(q_test, rewritten)
print(f'Candidates: {len(candidates)} (from 2x{RETRIEVAL_K})')
for r in candidates[:4]:
    print(f'  [{r["score"]:.4f}] [{r["topic_category"]}] {r["source"][:45]}')
    print(f'           {r["text"][:100]}...')
    print()

Candidates: 24 (from 2x20)
  [0.5913] [nutrition] Protein PTC 2022.pdf
           onsuming more protein than that resulted in no benefits for
strength development or muscle growth.

...

  [0.5713] [nutrition] Protein PTC 2022.pdf
           bstantially. Some people argue that AAS decrease
protein requirements because AAS increase nitrogen ...

  [0.5695] [nutrition] Protein PTC 2022.pdf
           bodyweight.

However, in obese individuals, 1.8 g/kg/d may be excessive. If you have a reasonably
re...

  [0.5648] [nutrition] Protein PTC 2022.pdf
           may also require extraordinarily high protein intakes.

Should you set protein relative to bodyweigh...



## 4. Answer Generation

In [12]:
SYSTEM_PROMPT = """Ти си персонален фитнес треньор и нутриционист. Работиш изцяло по методологията на Menno Henselmans.

ПРАВИЛА - следвай ги стриктно:
1. Отговаряй ВИНАГИ на БЪЛГАРСКИ
2. Ползвай КИЛОГРАМИ и метри, никога pounds или inches
3. Бъди ДИРЕКТЕН и ПРАКТИЧЕН - давай конкретни числа и препоръки
4. НЕ изнасяй лекции и НЕ цитирай проучвания, ако не са поискани
5. Говори като треньор, не като учебник - просто, ясно, приложимо
6. Ако нещо не е покрито в контекста, кажи го честно
7. Адаптирай отговора спрямо нивото и целта на потребителя
8. Посочвай винаги имената на упражненията единствено и САМО на английски език
9. Задължително базирай всичките си отговори САМО на ресурсите от курса

Контекст от курса на Henselmans (използвай САМО тази информация за факти):
{context}"""


@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def generate_answer(question: str, chunks: list[dict], history: list[dict] = []) -> str:
    """Generate Bulgarian answer with gpt-4o-mini using retrieved context."""
    context = '\n\n'.join(
        f'[{c["source"]}]\n{c["text"]}' for c in chunks
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT.format(context=context)}]

    for msg in history:
        messages.append({'role': msg['role'], 'content': msg['content']})

    messages.append({'role': 'user', 'content': question})

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=1200,
    )
    return response.choices[0].message.content


# Test
answer = generate_answer(q_test, candidates[:FINAL_K])
print(f'Q: {q_test}')
print()
print(answer)

Q: Колко протеин трябва да приемам за мускулна маса?

За оптимален растеж на мускулна маса, целта е да приемаш около 1.6 г протеин на килограм телесно тегло на ден. Ако си начинаещ или имаш мускулна памет, можеш да увеличиш до 2.2 г/kg. 

Пример: Ако тежиш 70 кг, целта ти е около 112 г протеин на ден (70 кг x 1.6 г). 

Не забравяй, че е важно да разпределиш приема на протеин равномерно през деня, като целиш поне 0.3 г/kg протеин в всяко хранене.


## 5. Full Pipeline

In [8]:
def ask(
    question: str,
    history: list[dict] = [],
    filter_meta: Optional[dict] = None,
) -> tuple[str, list[dict]]:
    """
    Full RAG pipeline: question → Bulgarian answer.

    Args:
        question: user question (typically Bulgarian)
        history: [{role: 'user'|'assistant', content: str}, ...]
        filter_meta: optional {'topic_category': 'nutrition'} etc.

    Returns:
        (answer, top_chunks)
    """
    rewritten = rewrite_query(question, history)
    candidates = dual_retrieve(question, rewritten)

    if filter_meta:
        filtered = [c for c in candidates
                    if all(c.get(k) == v for k, v in filter_meta.items())]
        candidates = filtered or candidates

    top_chunks = candidates[:FINAL_K]
    answer = generate_answer(question, top_chunks, history)
    return answer, top_chunks


print('ask() defined - 2 API calls per question')

ask() defined - 2 API calls per question


## 6. Tests

In [9]:
print('=' * 60)
print('TEST 1: Protein intake on a cut')
print('=' * 60)
answer, chunks = ask('Колко протеин трябва да приемам докато съм на дефицит?')
print(f'Sources: {[c["source"][:40] for c in chunks]}')
print()
print(answer)

TEST 1: Protein intake on a cut
Sources: ['Protein PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Protein PTC 2022.pdf', 'Protein PTC 2022.pdf']

Докато си на дефицит, препоръчителният прием на протеин е между 1.6 и 2.2 грама на килограм телесно тегло на ден. Ако целта ти е да запазиш мускулна маса, можеш да се целиш в около 2.0 г/kg. Например, ако тежиш 70 кг, това означава около 140 г протеин на ден. 

Важно е да разпределиш приема на протеин равномерно през деня, за да оптимизираш мускулната синтеза.


In [10]:
print()
print('=' * 60)
print('TEST 2: Training frequency')
print('=' * 60)
answer, chunks = ask('Колко пъти в седмицата трябва да тренирам всяка мускулна група?')
print()
print(answer)


TEST 2: Training frequency

Трябва да тренираш всяка мускулна група поне два пъти в седмицата. Ако обаче обемът на тренировките ти надвишава 16 серии на мускулна група, е препоръчително да тренираш мускулите поне три пъти в седмицата. За напреднали атлети, които работят с високи обеми, може да се обмисли дори ежедневна или двукратна тренировка на мускулите.


In [16]:
print()
print('=' * 60)
print('TEST 3: Multi-turn conversation')
print('=' * 60)
history = []

q1 = 'Какво е прогресивно претоварване?'
a1, _ = ask(q1)
history += [{'role': 'user', 'content': q1}, {'role': 'assistant', 'content': a1}]
print(f'Q1: {q1}')
print(f'A1: {a1[:300]}')
print()

q2 = 'Как да го приложа практически в тренировките си?'
a2, _ = ask(q2, history=history)
history += [{'role': 'user', 'content': q2}, {'role': 'assistant', 'content': a2}]
print(f'Q2: {q2}')
print(f'A2: {a2[:300]}')
print()

q3 = 'С колко килограма да увеличавам теглото всяка седмица във фитнеса?'
a3, _ = ask(q3, history=history)
print(f'Q3: {q3}')
print(f'A3: {a3[:300]}')


TEST 3: Multi-turn conversation
Q1: Какво е прогресивно претоварване?
A1: Прогресивното претоварване е принцип в тренировките, който означава постепенно увеличаване на стреса върху мускулите, за да се постигне адаптация и напредък. Това може да се реализира чрез увеличаване на теглото, с което тренираш, или чрез увеличаване на броя на повторенията, които изпълняваш с даде

Q2: Как да го приложа практически в тренировките си?
A2: Ето как можеш практически да приложиш прогресивното претоварване в тренировките си:

1. **Увеличаване на теглото**: Когато можеш да изпълниш максималния брой повторения в зададения диапазон (например 8-12 повторения), увеличи теглото с 2.5-5 кг на следващата тренировка. 

2. **Увеличаване на повторе

Q3: С колко килограма да увеличавам теглото всяка седмица във фитнеса?
A3: Ако си начинаещ, можеш да увеличаваш теглото с 1-2.5 кг на седмица. Това е достатъчно, за да предизвикаш мускулен растеж, без да рискуваш наранявания.

Ако си на по-високо ниво, увеличениет

In [13]:
print()
print('=' * 60)
print('TEST 4: Program design question')
print('=' * 60)
answer, chunks = ask(
    'Трябва ми примерна програма за напреднал състезател 3 пъти седмично',
    filter_meta={'is_case_study': True}
)
print(f'Case study sources:')
for c in chunks:
    print(f'  {c["source"][:55]}')
print()
print(answer)


TEST 4: Program design question
Case study sources:
  Case study average Joe program design.pdf
  Case study Chad BJJ.pdf
  Case study untrained.pdf
  Case study novice vegan.pdf
  Case study block periodization and mixing different goa

Ето примерна програма за напреднал състезател, която да се изпълнява 3 пъти седмично. Програмата е разделена на три дни, всеки от които включва основни упражнения за различни мускулни групи.

### Ден 1: Гърди и трицепс
1. **Bench Press**: 4 серии x 6-8 повторения
2. **Incline Dumbbell Press**: 3 серии x 8-10 повторения
3. **Chest Fly**: 3 серии x 10-12 повторения
4. **Triceps Dips**: 3 серии x 6-8 повторения
5. **Skull Crushers**: 3 серии x 8-10 повторения

### Ден 2: Гръб и бицепс
1. **Deadlift**: 4 серии x 6-8 повторения
2. **Pull-Ups**: 3 серии x 6-8 повторения
3. **Bent Over Row**: 3 серии x 8-10 повторения
4. **Seated Cable Row**: 3 серии x 10-12 повторения
5. **Barbell Bicep Curl**: 3 серии x 8-10 повторения

### Ден 3: Крака и рамене
1. **Squat

## 7. Summary

In [18]:
print('=' * 60)
print('  NOTEBOOK 06 - COMPLETE (OpenAI Stack)')
print('=' * 60)
print()
print('  Stack:')
print(f'  Embeddings: text-embedding-3-large (3072d)')
print(f'  LLM: gpt-4o-mini')
print(f'  Retrieval: FAISS IndexFlatIP (exact cosine)')
print()
print('  System prompt: direct, practical, Bulgarian,')
print('  kilograms, coach-style - no unnecessary citations')

  NOTEBOOK 06 - COMPLETE (OpenAI Stack)

  Stack:
  Embeddings: text-embedding-3-large (3072d)
  LLM: gpt-4o-mini
  Retrieval: FAISS IndexFlatIP (exact cosine)

  System prompt: direct, practical, Bulgarian,
  kilograms, coach-style - no unnecessary citations
